## Imports

In [1]:
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from pathlib import Path
import numpy as np
import json

## Define Project Paths

In [2]:
PROJECT_PATH = Path.cwd().parent

QDRANT_PATH = (PROJECT_PATH / "data" / "processed" / "vector_store" / "qdrant")

COLLECTION_NAME = "financial_policies"

print("Project path:", PROJECT_PATH)
print("Qdrant path:", QDRANT_PATH)
print("Collection:", COLLECTION_NAME)

Project path: /Users/pushkarkamat/Desktop/financial-rag
Qdrant path: /Users/pushkarkamat/Desktop/financial-rag/data/processed/vector_store/qdrant
Collection: financial_policies


In [3]:
client = QdrantClient(path=str(QDRANT_PATH))
print("Qdrant Client Connected")

Qdrant Client Connected


## Verify the collection

In [4]:
collection_info = client.get_collection(COLLECTION_NAME)

print("Collection:", COLLECTION_NAME)
print("Vectors stored:", collection_info.points_count)

Collection: financial_policies
Vectors stored: 295


## Load BGE-large

In [5]:
EMBED_MODEL = "BAAI/bge-large-en-v1.5"
embedding_model = SentenceTransformer(EMBED_MODEL)

print("Embedding model loaded.")
print("Embedding dimension:", embedding_model.get_embedding_dimension())

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Embedding model loaded.
Embedding dimension: 1024


## Create first query

In [6]:
query = "What is the maximum amount that requires enhanced credit approval?"
print("Query:", query)

Query: What is the maximum amount that requires enhanced credit approval?


## Convert Query into Embeddings

In [7]:
embedded_query = embedding_model.encode(
    query,
    normalize_embeddings=True
)

print("Query embedding shape:", embedded_query.shape)
print("Query embedding dtype:", embedded_query.dtype)
print("Query embedding norm:", np.linalg.norm(embedded_query))

Query embedding shape: (1024,)
Query embedding dtype: float32
Query embedding norm: 1.0


## Retrieve the most relevant chunks

In [9]:
search_results = client.query_points(
    collection_name=COLLECTION_NAME,
    query=embedded_query.tolist(),
    limit=5,
    with_payload=True,
).points

print("Retrieved chunks:", len(search_results))

Retrieved chunks: 5


## Inspect what Qdrant retrieved

In [10]:
for i, result in enumerate(search_results, start=1):
    print("=" * 75)
    print(f"Result: {i}")
    print(f"Score: {result.score}")
    print(f"Chunk ID: {result.payload.get('chunk_id')}")
    print(f"Document: {result.payload.get('document')}")
    print("Text:")
    print(result.payload.get("text"))
    print()

Result: 1
Score: 0.6610575981423698
Chunk ID: chunk_00000043
Document: 01_Credit_Risk_Policy_CRP-001_v2.0.docx
Text:
Section: 13. Credit Approval and Delegated Authority

Credit decisions must be made by an officer or committee holding sufficient delegated authority for the product, amount, risk grade and exception status. Authority is personal to the designated role and remains subject to restrictions stated in CDAS-007.

Section: 13. Credit Approval and Delegated Authority

Credit Risk expects documented judgement, clear ownership and evidence that can be reconstructed after the decision. The requirements in this section apply unless a documented product rule, approved exception, or later effective instruction expressly provides otherwise.

Section: 13. Credit Approval and Delegated Authority

Control requirements

Section: 13. Credit Approval and Delegated Authority

• CA1 may approve standard personal loans up to ₹10 lakh but is limited to ₹8 lakh for new-vehicle finance, ₹6 lakh f

----

## Testing Queries

In [11]:
test_queries = [
    "What is the maximum personal loan amount that CA2 can approve?",
    "How much can CA2 authorize for a standard personal loan?",
    "What is Northstar's maximum credit-card limit for students?"
]

for query in test_queries:
    print("=" * 80)
    print("QUERY:", query)

    query_embedding = embedding_model.encode(
        query,
        normalize_embeddings=True
    )

    results = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_embedding.tolist(),
        limit=3,
        with_payload=True,
    ).points

    for i, result in enumerate(results, start=1):
        print(f"\nResult {i}")
        print("Score:", round(result.score, 4))
        print("Chunk:", result.payload.get("chunk_id"))
        print("Document:", result.payload.get("document"))
        print("Text:", result.payload.get("text")[:500])

QUERY: What is the maximum personal loan amount that CA2 can approve?

Result 1
Score: 0.7522
Chunk: chunk_00000159
Document: 03_Credit_Delegated_Authority_Schedule_CDAS-007_v2.1.docx
Text: Section: 4. Personal Loan Authority

Standard personal loans retain the general CA1 and CA2 retail limits but remain subject to the product cap and any elevated authority triggered by risk or exception status.

Section: 4. Personal Loan Authority

Authority administration treats the controlled schedule as the source of decision rights; workflow permissions are supporting evidence only. The requirements in this section apply unless a documented product rule, approved exception, or later effective

Result 2
Score: 0.7426
Chunk: chunk_00000105
Document: 02_Credit_Exception_Procedure_CEP-006_v1.3.docx
Text: • A personal-loan score of 620–639 is an E2 exception and requires at least CA3 approval in addition to manual underwriting.

Section: 7. Credit-Score and Thin-File Exceptions

• A score below 620 is

---

## def retrieval

In [12]:
def retrieve_documents(query, top_k=5):
    """
    Retrieve the most relevant policy chunks from Qdrant.
    """

    query_embedding = embedding_model.encode(
        query,
        normalize_embeddings=True
    )

    results = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_embedding.tolist(),
        limit=top_k,
        with_payload=True,
    ).points

    return results

### testing 

In [13]:
results = retrieve_documents(
    "How much can CA2 authorize for a standard personal loan?",
    top_k=5
)

for i, result in enumerate(results, start=1):
    print("=" * 70)
    print(f"Result {i}")
    print("Score:", round(result.score, 4))
    print("Chunk:", result.payload.get("chunk_id"))
    print("Document:", result.payload.get("document"))
    print()

Result 1
Score: 0.7696
Chunk: chunk_00000159
Document: 03_Credit_Delegated_Authority_Schedule_CDAS-007_v2.1.docx

Result 2
Score: 0.7533
Chunk: chunk_00000105
Document: 02_Credit_Exception_Procedure_CEP-006_v1.3.docx

Result 3
Score: 0.7447
Chunk: chunk_00000199
Document: 03_Credit_Delegated_Authority_Schedule_CDAS-007_v2.1.docx

Result 4
Score: 0.7338
Chunk: chunk_00000043
Document: 01_Credit_Risk_Policy_CRP-001_v2.0.docx

Result 5
Score: 0.7262
Chunk: chunk_00000172
Document: 03_Credit_Delegated_Authority_Schedule_CDAS-007_v2.1.docx

